# 🎬 Netflix Pakistan | In-depth Exploratory Analysis with Plotly

**A historical content-catalogue study, not audience analytics.**  Inspired by the visual storytelling approach of [In-Depth Analysis of Netflix with Plotly](https://www.kaggle.com/code/rushikeshdane20/in-depth-analysis-of-netflix-with-plotly). All code, selection logic, chart styling and commentary here are original.

**Research questions**
1. How many titles credit Pakistan as a production country?
2. What are their formats, genres, ratings and release patterns?
3. When did Netflix add those titles in the historical source dataset?
4. How does Pakistan compare with other South Asian production countries?

**Important:** the source ends in September 2021. Production country is not the availability territory; this notebook does not estimate viewership, subscriber counts or current availability.

In [ ]:
from pathlib import Path
import sys

# Allows running the notebook from notebooks/ or the repository root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio
from src.data import load_catalog, pakistan_titles, snapshot, regional_counts, genre_counts
from src.charts import (type_mix, yearly_additions, genres_bar, ratings_bar,
                        release_timeline, regional_comparison, coproductions, lead_time)

pio.renderers.default = "notebook_connected"
full = load_catalog(ROOT / "data" / "netflix_titles.csv.gz")
pk = pakistan_titles(full)
stats = snapshot(full)
stats

## 01 — Data quality and selection

We keep Pakistan **only** when the exact token appears in the comma-delimited production-country field, so `Pakistan`, `India, Pakistan`, and `Pakistan, United States` count but unrelated strings cannot accidentally match. Country names that are missing are unknown, not zero Pakistan affiliations.

The raw source contains 8,807 title records. Every chart uses unique title counts except explicitly multi-label genre or country attributions.

In [ ]:
pd.DataFrame({
    "field": ["country", "date_added", "listed_in", "rating", "director", "cast"],
    "missing_source_records": [int(full[field].isna().sum()) for field in
                               ["country", "date_added", "listed_in", "rating", "director", "cast"]]
})

In [ ]:
pd.DataFrame({"metric": list(stats), "value": list(stats.values())})

## 02 — Film vs series

Pakistan has **20 films and 4 TV series** in this historical dataset. This measures title records, not hours watched, catalogue impressions or popularity.

In [ ]:
type_mix(pk).show()

## 03 — When the catalogue added Pakistan-linked titles

Added dates reflect *when the dataset says a title was added to Netflix*, not when it was made or watched. We group by `added_year`, not `release_year`.

In [ ]:
yearly_additions(pk).show()

## 04 — Category analysis

Genres are **multi-label**: a drama can also be a romantic movie or an international movie. The summed genre counts therefore exceed 24 and should not be interpreted as shares of a mutually exclusive population.

In [ ]:
genre_counts(pk, top_n=12)

In [ ]:
genres_bar(pk).show()

## 05 — Rating classes

The source's `rating` column contains content-maturity classifications (for example `TV-14`), **not IMDb ratings** or reviewers' scores.

In [ ]:
ratings_bar(pk).show()

## 06 — Release years and time to catalogue addition

`release_year` describes a work's release; `added_year` is a later catalogue event. Their difference is an approximation measured in whole years, not a precise delay in days.

In [ ]:
release_timeline(pk).show()

In [ ]:
lead_time(pk).show()

## 07 — Co-productions

A multi-country credit is counted as a co-production for these descriptive comparisons. The CSV does not establish ownership, financing shares, or which partner led a production.

In [ ]:
coproductions(pk).show()

In [ ]:
pk.loc[pk.is_coproduction, ["title", "country"]].sort_values("title")

## 08 — South Asian comparison

Production countries are **credited categories**. Multi-country titles can appear in more than one country's count. Use the log-scale chart to see smaller country counts without visually erasing them next to India's larger count. These are not Netflix user-location statistics.

In [ ]:
regional_counts(full).sort_values("titles", ascending=False)

In [ ]:
regional_comparison(full).show()

## 09 — Source-level title audit

Inspect every Pakistan-linked title and verify the earlier calculations. Export an auditable flat table for independent review.

In [ ]:
columns = ["title", "type", "release_year", "added_year", "rating", "country", "listed_in"]
view = pk[columns].sort_values("title")
view

In [ ]:
# Optional export, writes only after you uncomment:
# view.to_csv(ROOT / "reports" / "pakistan_titles.csv", index=False)
print(f"Audited {len(view)} Pakistan-linked titles out of {len(full)} global records")

## 10 — Conclusion and limits

The source supports a descriptive account of Pakistan's **production credits** in Netflix's global historical title catalogue. It **does not** indicate whether any of these titles were available to a viewer in Pakistan, how many people watched them, their audience sentiment, or how many were on Netflix in 2026.

The Pakistan sample is **24** records. Report observed counts, avoid predictive claims, and disclose the source end date.

**References**
- [Netflix Movies and TV Shows dataset](https://www.kaggle.com/datasets/shivamb/netflix-shows)
- [Plotly-focused Kaggle analysis — visualization inspiration](https://www.kaggle.com/code/rushikeshdane20/in-depth-analysis-of-netflix-with-plotly)
